<a href="https://colab.research.google.com/github/ttktjmt/mjswan/blob/claude%2Fload-onnx-from-wandb-9Och3/examples/mjlab/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train policies for mjlab default tasks

Train policy with my recommended parameters for each mjlab default task using a free GPU from google colab. 
Those policies will be used by a [defaults.py](defaults.py)

# Train policies for mjlab default tasks

Train a policy with recommended hyperparameters for each mjlab default task using a free Colab GPU.
Trained policies are used in the [defaults.py](defaults.py).

## Setup

In [ ]:
# Install mjlab from source
import os
import subprocess

try:
    import google.colab  # type: ignore # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    if not os.path.isdir("mjlab"):
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/mujocolab/mjlab.git"], check=True
        )
    os.chdir("mjlab")
    subprocess.run(["uv", "pip", "install", "--system", "-e", ".", "-q"], check=True)
else:
    mjlab_dir = os.path.join(os.getcwd(), "mjlab")
    if not os.path.isdir(mjlab_dir):
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/mujocolab/mjlab.git", mjlab_dir],
            check=True,
        )
    subprocess.run(["uv", "sync"], cwd=mjlab_dir, check=True)

In [ ]:
!wandb login

In [3]:
!python -m mjlab.scripts.list_envs

+---------------------------------------------------------+
|             Available Environments in mjlab             |
+----+----------------------------------------------------+
| #  | Task ID                                            |
+----+----------------------------------------------------+
| 1  | Mjlab-Cartpole-Balance                             |
| 2  | Mjlab-Cartpole-Swingup                             |
| 3  | Mjlab-Lift-Cube-Yam                                |
| 4  | Mjlab-Lift-Cube-Yam-Depth                          |
| 5  | Mjlab-Lift-Cube-Yam-Rgb                            |
| 6  | Mjlab-Multi-Cube-Seg-Yam                           |
| 7  | Mjlab-Tracking-Flat-Unitree-G1                     |
| 8  | Mjlab-Tracking-Flat-Unitree-G1-No-State-Estimation |
| 9  | Mjlab-Velocity-Flat-Unitree-G1                     |
| 10 | Mjlab-Velocity-Flat-Unitree-Go1                    |
| 11 | Mjlab-Velocity-Rough-Unitree-G1                    |
| 12 | Mjlab-Velocity-Rough-Unitree-Go1 

In [4]:
import os
import subprocess
import sys

TASK_RUN_ID_MAP = {
    "Mjlab-Cartpole-Balance": "cartpole-balance",
    "Mjlab-Cartpole-Swingup": "cartpole-swingup",
    "Mjlab-Lift-Cube-Yam": "lift-cube-yam",
    "Mjlab-Velocity-Flat-Unitree-G1": "vel-flat-g1",
    "Mjlab-Velocity-Flat-Unitree-Go1": "vel-flat-go1-v3",
    "Mjlab-Velocity-Rough-Unitree-G1": "vel-rough-g1",
    "Mjlab-Velocity-Rough-Unitree-Go1": "vel-rough-go1",
}


def run_train(task_name: str, num_envs: int, max_iter: int, save_int: int):
    # os.environ["WANDB_RUN_ID"] = TASK_RUN_ID_MAP[task_name]

    process = subprocess.Popen(
        [
            "python",
            "-m",
            "mjlab.scripts.train",
            task_name,
            "--env.scene.num-envs",
            str(num_envs),
            "--agent.max-iterations",
            str(max_iter),
            "--agent.save-interval",
            str(save_int),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()

    process.wait()

In [ ]:
def run_train_from_checkpoint(
    task_name: str, num_envs: int, max_iter: int, save_int: int, wandb_run_path: str
):
    process = subprocess.Popen(
        [
            "python",
            "-m",
            "mjlab.scripts.train",
            task_name,
            "--env.scene.num-envs",
            str(num_envs),
            "--agent.max-iterations",
            str(max_iter),
            "--agent.save-interval",
            str(save_int),
            "--wandb-run-path",
            wandb_run_path,
            "--agent.resume",
            "True",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()

    process.wait()

## Mjlab-Cartpole-Balance

In [ ]:
run_train(
    task_name="Mjlab-Cartpole-Balance",
    num_envs=1_024,
    max_iter=101,
    save_int=20,
)

## Mjlab-Cartpole-Swingup

In [ ]:
run_train(
    task_name="Mjlab-Cartpole-Swingup",
    num_envs=1_024,
    max_iter=101,
    save_int=20,
)

## Mjlab-Lift-Cube-Yam

In [ ]:
run_train(
    task_name="Mjlab-Lift-Cube-Yam",
    num_envs=4_096,
    max_iter=3_001,
    save_int=200,
)

## Mjlab-Velocity-Flat-Unitree-G1

In [ ]:
run_train(
    task_name="Mjlab-Velocity-Flat-Unitree-G1",
    num_envs=4_096,
    max_iter=2_001,
    save_int=200,
)

## Mjlab-Velocity-Flat-Unitree-Go1

In [ ]:
run_train(
    task_name="Mjlab-Velocity-Flat-Unitree-Go1",
    num_envs=4_096,
    max_iter=1_201,
    save_int=100,
)

## Mjlab-Velocity-Rough-Unitree-G1

In [ ]:
run_train_from_checkpoint(
    task_name="Mjlab-Velocity-Rough-Unitree-G1",
    num_envs=2_048,
    max_iter=20_001,
    save_int=500,
    wandb_run_path="ttktjmt-org/mjlab/mowqlkd5",
)

In [ ]:
run_train(
    task_name="Mjlab-Velocity-Rough-Unitree-G1",
    num_envs=2_048,  # half size due to GPU RAM limit
    max_iter=20_001,
    save_int=500,
)